# Detection of Spatial Signatures in Winterthur

This notebook contains complete code required for a detection of spatial signatures in Winterthur, Switzerland, limited by its municipal administrative boundary — not the canton.

It follows the same four-stage pipeline (form → function → clustering → figures) as `zurich.ipynb` (the canton-wide run), with several deliberate changes on top of just shrinking the extent:

- **Scope**: Winterthur municipality, geocoded with the structured Nominatim query (`{"city": "Winterthur", "country": "Switzerland"}`).
- **Water as an enclosure barrier**: water boundaries are passed as `additional_barriers` to `momepy.enclosures`, so waterways create their own barrier-bounded enclosures with no tessellation inside.
- **Parks**: the inherited city workflow and its data-processing logic are retained unchanged.
- **Land use**: kept at OSM's full `landuse` category granularity — no collapsing into a handful of coarse buckets.
- **Population**: uses the Canton of Zürich official hectare-grid population statistics (`stat_bevoelkerung_ha`), area-interpolated onto tessellation cells.
- **New function layers**: employment/jobs (`stat_beschaeftigte_ha`), building stock — age/type/dwellings (`stat_gebaeude_ha`), public-transport quality & spatial-planning typology (`afv_gv_ew_oevgk`), and OSM street-network betweenness centrality (a whole-network "importance for movement" measure, distinct from the local morphometric street characters already in form).
- **Form vs. function**: form stays exclusively built-form/morphometric (building shape, tessellation/enclosure geometry, local street-network structure). Function absorbs every other layer, including all the new ones above.
- **NDVI and night lights** are left as `TODO` stubs, same as the canton notebook (new imagery providers/deps, out of scope here).

## Environment

Same pinned `.venv` as `zurich.ipynb` (Python 3.9, `geopandas==0.13.2`, `osmnx==1.9.4`, `momepy==0.6.0`, `numpy<2` — see `code/requirements_zurich.txt`). No new dependencies: the new data sources are fetched as plain WFS GeoJSON over `requests`, and area-weighted interpolation is a small addition to `utils/tobler.py` (no external `tobler` package, which isn't installed).

## Contents

- 1. Analysis of form
    - 1.1. Data retrieval
        - 1.1.1. Building footprints
        - 1.1.2. Street network
        - 1.1.3. Spatial barriers (water, railway, city boundary)
    - 1.2. Data pre-processing
        - 1.2.1. Footprints checks and cleaning
        - 1.2.2. Barriers checks and cleaning
    - 1.3. Generation of geographies
        - 1.3.1. Enclosures
        - 1.3.2. Enclosed tessellation
    - 1.4. Morphometric analysis
        - 1.4.1. Primary morphometric characters
        - 1.4.2. Contextualisation
- 2. Analysis of function
    - 2.1. Eating establishments
    - 2.2. Supermarkets
    - 2.3. Land use (full OSM granularity)
    - 2.4. Parks
    - 2.5. Historical monuments
    - 2.6. Street network importance (betweenness centrality)
    - 2.7. Population (official hectare grid)
    - 2.8. Employment (official hectare grid)
    - 2.9. Building stock (official hectare grid)
    - 2.10. Public transport quality & planning typology
    - 2.11. NDVI (TODO)
    - 2.12. Night lights (TODO)
- 3. Cluster analysis
    - 3.1. Form
    - 3.2. Function
    - 3.3. Spatial Signatures
- 4. Figures
- 5. Cluster summary

All packages and custom modules are imported below.


In [ ]:
import os
import pickle
import warnings

import geopandas as gpd
import dask_geopandas
import libpysal
import momepy
import networkx as nx
import numpy as np
import osmnx as ox
import pandana
import pandas as pd
import rasterstats
import requests
import rioxarray
import scipy
import utils
import utils.tobler
import xarray
from clustergram import Clustergram
from dask.distributed import Client, LocalCluster
from download import download
from shapely.geometry import box, mapping, shape
from shapely.ops import polygonize
from sklearn import preprocessing
from sklearn.cluster import MiniBatchKMeans
from tqdm import tqdm
from utils.tobler import area_max, area_interpolate
from utils.dask_geopandas import dask_dissolve

import urbangrammar_graphics as ugg
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import contextily

Most of our data will be stored as geospatial parquet files. As in the Singapore notebook, we filter the associated (harmless) warning.

In [ ]:
warnings.filterwarnings('ignore', message='.*initial implementation of Parquet.*')


First we specify a path to a folder for data storage (a local folder within the repository, since the original `urbangrammar_samba` network share isn't available on this machine) and the structured OSM place query for the city.

In [ ]:
folder = "../data/winterthur/"
os.makedirs(folder, exist_ok=True)

# Structured query, not a free-text place name: "Zürich, Switzerland" is
# ambiguous between the city and the canton (both are named "Zürich" in OSM).
# This resolves to OSM relation 1682248 (admin_level=8, the municipal boundary).
place = {"city": "Winterthur", "country": "Switzerland"}

ox.settings.timeout = 1800
ox.settings.memory = None

## 1. Analysis of form

First step towards spatial signatures is an analysis of urban form to capture its patterns.

### 1.1. Data retrieval

All data used in this section are obtained from OpenStreetMap.

#### 1.1.1 Building footprints

Building footprints are obtained from OSM using OSMnx, exactly as in the Singapore notebook.

If this single Overpass query times out or is rejected for being too large (a real risk over a large study area, though much less so for a ~92 km² city than the ~1'700 km² canton), a more robust fallback is to download a regional `.osm.pbf` extract (e.g. from Geofabrik) and clip/parse it locally with `pyrosm` or `osmium` instead of querying the live Overpass API. That path isn't implemented here to keep parity with the Singapore notebook's approach.

In [ ]:
%%time
path = folder + 'footprints_osm.pq'
if not os.path.exists(path):
    footprints = ox.geometries_from_place(place, tags={'building': True})
    footprints_proj = ox.projection.project_gdf(footprints)
    cols = [c for c in ['building', 'building:levels', 'geometry'] if c in footprints_proj.columns]
    footprints_proj[cols].to_parquet(path)

#### 1.1.2 Street network

Street network is obtained from OSM using OSMnx, and the same custom tag filter used for Singapore. We get a graph, project it, turn it into an undirected one and export it to a GeoDataFrame. Then we make sure `highway` is a string column so it can be saved to file.

Note: `network_type='all'` is used instead of the deprecated `'all_private'` alias (renamed in osmnx, though still accepted in 1.9.x with a warning).

In [ ]:
%%time
path = folder + 'roads_osm.pq'
if not os.path.exists(path):
    type_filter = '["highway"~"living_street|motorway|motorway_link|pedestrian|primary|primary_link|residential|secondary|secondary_link|service|tertiary|tertiary_link|trunk|trunk_link"]'

    streets_graph = ox.graph_from_place(place, network_type='all', custom_filter=type_filter, retain_all=True)
    streets_graph = ox.projection.project_graph(streets_graph)
    gdf = ox.graph_to_gdfs(ox.get_undirected(streets_graph), nodes=False, edges=True, node_geometry=False, fill_edge_geometry=True)
    gdf.highway = gdf.highway.astype(str)
    gdf[['highway', 'geometry']].to_parquet(path)

#### 1.1.3. Spatial barriers

Identification of enclosures should optimally contain a variety of barriers, not only the street network. Therefore, we also download water bodies and railway geometry from OSM, and the municipal administrative boundary itself (used below as the outer `limit`, since &mdash; unlike Singapore &mdash; Zürich is landlocked and has no coastline to polygonize into an outline).

##### Water

In [ ]:
%%time
path = folder + 'water_osm.pq'
if not os.path.exists(path):
    tags = {'natural': ['water', 'coastline', 'spring']}
    gdf = ox.geometries_from_place(place, tags)
    gdf[['natural', 'geometry']].to_parquet(path)

##### Railway

Then we get railway and filter out the proper geometry type (we want LineStrings representing railway tracks) and remove tunnels (those are not spatial barriers).

In [ ]:
path = folder + 'railway_osm.pq'
if not os.path.exists(path):
    tags = {'railway': True}
    gdf = ox.geometries_from_place(place, tags)
    gdf = gdf[gdf.geom_type == 'LineString']
    gdf = gdf[gdf.get('tunnel') != 'yes'] if 'tunnel' in gdf.columns else gdf
    gdf[['railway', 'geometry']].to_parquet(path)

##### City boundary

Used as the outer `limit` for enclosures below (Singapore's notebook instead polygonized water/coastline into an island outline &mdash; not applicable to a landlocked city).

In [ ]:
path = folder + "admin_osm.pq"
if not os.path.exists(path):
    admin = ox.geocode_to_gdf(place)
    admin.to_parquet(path)
else:
    admin = gpd.read_parquet(path)

# Sanity check: the city of Zürich is ~92 km²; the canton is ~1729 km².
# If a future Nominatim change resolves `place` to the wrong entity, fail loudly
# here instead of silently re-running the whole pipeline at canton scale.
_admin_area_km2 = ox.projection.project_gdf(admin).area.sum() / 1e6
print(f"Geocoded area: {_admin_area_km2:.1f} km²")
assert 50 < _admin_area_km2 < 150, (
    f"Geocoded area {_admin_area_km2:.1f} km² is outside the plausible range "
    "for the city of Zürich — check that `place` resolved to the city, not the canton."
)

### 1.2. Data pre-processing

Before we can analyse form using the data we just downloaded, we need to ensure that they are in a proper shape.

#### 1.2.1. Footprints checks and cleaning

Let's start with building footprints.

In [ ]:
buildings = gpd.read_parquet(folder + 'footprints_osm.pq')


OSM data tend to be messy, so we need to check validity and geometry types.

In [ ]:
buildings.geom_type.value_counts()


We explode geometries and keep only polygons.

In [ ]:
buildings = buildings.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True)
buildings = buildings[buildings.geom_type == 'Polygon']


We can then use `momepy.CheckTessellationInput()` to check for potential issues which may arise during enclosed tessellation.

In [ ]:
%time check = momepy.CheckTessellationInput(buildings)


As in the Singapore notebook, we drop collapsed and overlapping geometries rather than manually fixing them.

In [ ]:
buildings = buildings.drop(check.collapse.index.union(check.overlap.index))


In [ ]:
buildings.shape


In [ ]:
buildings.is_valid.all()


We assign a unique ID to each remaining building and save them to a file.

In [ ]:
buildings["uID"] = range(len(buildings))
%time buildings.to_parquet(folder + "buildings.pq")


#### 1.2.2. Barriers checks and cleaning

Let's now check our barriers and close potential unwanted gaps to generate clean enclosures.

In [ ]:
roads = gpd.read_parquet(folder + "roads_osm.pq").to_crs(buildings.crs)


In [ ]:
roads.geom_type.unique()


In [ ]:
railway = gpd.read_parquet(folder + "railway_osm.pq").to_crs(buildings.crs)


In [ ]:
railway.geom_type.unique()


Roads seem fine, similarly to railway. The only step we do is extension of railway lines to snap to roads.

In [ ]:
extended_railway = momepy.extend_lines(railway, 30, target=roads, extension=.1)


In [ ]:
water = gpd.read_parquet(folder + "water_osm.pq").to_crs(buildings.crs)


In [ ]:
water.geom_type.value_counts()


We create LineString geometry out of polygons, so lakes and rivers act as barriers rather than areas.

In [ ]:
mask = water.geom_type.isin(["MultiPolygon", "Polygon"])


In [ ]:
# Union first so adjacent/overlapping water polygons (e.g. a lake split into
# multiple OSM ways) don't leave internal slivers when we take the boundary.
from shapely.ops import unary_union
water_union = gpd.GeoSeries([unary_union(water.loc[mask, "geometry"])], crs=water.crs)
water.loc[mask, "geometry"] = water[mask].boundary
water = pd.concat(
    [water[~mask], gpd.GeoDataFrame({"natural": ["water"]}, geometry=water_union.boundary, crs=water.crs)],
    ignore_index=True,
)

We read the city boundary and reproject it to match. This is the `limit` used for enclosures below — the outer extent of the study area.

In [ ]:
admin = gpd.read_parquet(folder + "admin_osm.pq").to_crs(buildings.crs)
limit = admin.geometry.unary_union


### 1.3. Generation of geographies

Now we create the geographies we will be using in the analysis: enclosures and enclosed tessellation.

#### 1.3.1. Enclosures

Enclosures require spatial barriers &mdash; roads, water and railway (using the extended one from above) &mdash; limited by the city boundary.

In [ ]:
%%time
path = folder + "enclosures.pq"
if os.path.exists(path):
    enclosures = gpd.read_parquet(path)
else:
    enclosures = momepy.enclosures(roads, limit=gpd.GeoSeries([limit], crs=buildings.crs), additional_barriers=[extended_railway, water])
    enclosures.to_parquet(path)

In [ ]:
enclosures.is_valid.all()


Water was passed to `momepy.enclosures` as an `additional_barrier`, so Lake Zürich and the Limmat/Sihl rivers should already be cutting their own barrier-bounded enclosure(s) with no tessellation inside (no buildings sit in the water). Verify this visually before moving on.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
enclosures.plot(ax=ax, facecolor="none", edgecolor="grey", linewidth=0.3)
water.plot(ax=ax, color="tab:blue", linewidth=1.5)
ax.set_title("Enclosures (grey) with water barriers (blue) overlaid")
ax.set_axis_off()

#### 1.3.2. Enclosed tessellation

With enclosures and buildings, we can now create the enclosed tessellation. To have better control of the dask cluster, we create it directly (enclosed tessellation is a dask-parallelised function). Adjust `n_workers` to your machine.

In [ ]:
# Skip starting a dask cluster entirely if tessellation.pq is already cached --
# LocalCluster startup has proven flaky in this environment (Windows + zmq/asyncio),
# and it is only needed for the momepy.Tessellation call in the next cell.
if os.path.exists(folder + "tessellation.pq"):
    client = None
    print("tessellation.pq already exists; skipping dask cluster startup.")
else:
    client = Client(LocalCluster(n_workers=8))
client


In [ ]:
%%time
path = folder + "tessellation.pq"
if os.path.exists(path):
    tessellation = gpd.read_parquet(path)
else:
    tess = momepy.Tessellation(buildings, 'uID', enclosures=enclosures)
    tessellation = tess.tessellation
    tessellation['tID'] = range(len(tessellation))
    tessellation.to_parquet(path)

Now we can assign a unique ID to each tessellation cell and save it.

### 1.4. Morphometric analysis

Morphometric analysis first measures primary characters, which are then contextualised to capture the tendencies within the local context of each tessellation cell.

#### 1.4.1. Primary morphometric characters

For the analysis we need four input layers representing urban form: buildings, streets, enclosed tessellation, and enclosures.

In [ ]:
tess = gpd.read_parquet(folder + "tessellation.pq")
blg = gpd.read_parquet(folder + "buildings.pq")
streets = gpd.read_parquet(folder + 'roads_osm.pq')
enclosures = gpd.read_parquet(folder + "enclosures.pq")


In [ ]:
tess.crs == blg.crs == streets.crs == enclosures.crs


In many cases, we will need to relate buildings to tessellation cells. The best way to ensure a proper linkage is to combine both GeoDataFrames into a single one with two named geometry columns. Not every cell has an assigned building.

In [ ]:
tess = tess.rename_geometry("tessellation").merge(blg[['uID', 'geometry']].rename_geometry("buildings"), on='uID', how='left')


In [ ]:
tess


From the combined GeoDataFrame, we can create one with buildings as the active geometry, dropping rows without a building.

In [ ]:
blg = tess.set_geometry('buildings').dropna()


Now we can measure morphometric characters. Refer to the [momepy documentation](https://docs.momepy.org) for details on each one. Each result is assigned as a new column. (This mirrors the Singapore notebook's cell exactly, using `momepy.*` directly &mdash; the custom `utils.momepy` wrappers this repository used in 2021 have since been merged upstream into momepy itself.)

In [ ]:
%time blg['sdbAre'] = momepy.Area(blg).series
%time blg['sdbPer'] = momepy.Perimeter(blg).series
%time blg['sdbCoA'] = momepy.CourtyardArea(blg, 'sdbAre').series

%time blg['ssbCCo'] = momepy.CircularCompactness(blg, 'sdbAre').series
%time blg['ssbCor'] = momepy.Corners(blg).series
%time blg['ssbSqu'] = momepy.Squareness(blg).series
%time blg['ssbERI'] = momepy.EquivalentRectangularIndex(blg, 'sdbAre', 'sdbPer').series
%time blg['ssbElo'] = momepy.Elongation(blg).series
%time cencon = momepy.CentroidCorners(blg)
blg['ssbCCM'] = cencon.mean
blg['ssbCCD'] = cencon.std

%time blg['stbOri'] = momepy.Orientation(blg).series

%time tess['stcOri'] = momepy.Orientation(tess).series

tess = tess.merge(blg[['tID', 'stbOri']], on='tID', how='left')

%time tess['stbCeA'] = (tess['stbOri'] - tess['stcOri']).abs()

%time tess['sdcLAL'] = momepy.LongestAxisLength(tess).series
%time tess['sdcAre'] = momepy.Area(tess).series
%time tess['sscCCo'] = momepy.CircularCompactness(tess, 'sdcAre').series
%time tess['sscERI'] = momepy.EquivalentRectangularIndex(tess, 'sdcAre').series

%time tess['sicCAR'] = tess.buildings.area / tess['sdcAre']
%time blg["mtbSWR"] = momepy.SharedWallsRatio(blg).series


Some characters require spatial weights matrices. We generate Queen contiguity based on the enclosed tessellation.

In [ ]:
%time queen_1 = libpysal.weights.contiguity.Queen.from_dataframe(tess, ids="tID", geom_col='tessellation')
%time tess["mtbAli"] = momepy.Alignment(tess.set_geometry("buildings"), queen_1, "tID", "stbOri").series
%time tess["mtbNDi"] = momepy.NeighborDistance(tess.set_geometry("buildings"), queen_1, "tID").series
%time tess["mtcWNe"] = momepy.Neighbors(tess, queen_1, "tID", weighted=True).series
%time tess["mdcAre"] = momepy.CoveredArea(tess, queen_1, "tID").series
%time blg_q1 = libpysal.weights.contiguity.Queen.from_dataframe(blg, geom_col='buildings', silence_warnings=True)

%time blg["ldbPWL"] = momepy.PerimeterWall(blg, blg_q1).series
%time blg["libNCo"] = momepy.Courtyards(blg, spatial_weights=blg_q1).series
%time queen_3 = momepy.sw_high(k=3, weights=queen_1)
%time tess['ltbIBD'] = momepy.MeanInterbuildingDistance(tess.set_geometry('buildings'), queen_1, 'tID', queen_3).series


We also need to link tessellation cells to streets, based on intersection. This gives us the ratio if the tessellation intersects more than one street segment. From these ratios we get the primary link (the one intersecting the most). Since we need these links on both tessellation and buildings, we merge the result into `blg`.

In [ ]:
# get nid and primary nid here
%time links = momepy.get_network_ratio(tess, streets)
tess[['edgeID_keys', 'edgeID_values']] = links
keys = tess.edgeID_values.apply(lambda a: np.argmax(a))
tess['edgeID_primary'] = [inds[i] for inds, i in zip(tess.edgeID_keys, keys)]
blg = blg.merge(tess[['tID', 'edgeID_primary']], on='tID', how='left')
streets['edgeID_primary'] = range(len(streets))


In [ ]:
# Checkpoint blg/tess/streets/queen_1 here so a crash past this point doesn't
# require re-running the whole morphometric section above. Only *restore* from
# it if it already exists (a fresh data/zurich_city/ folder won't have one yet).
_ckpt = {
    "blg": folder + "blg_checkpoint2.pq",
    "tess": folder + "tess_checkpoint2.pq",
    "streets": folder + "streets_checkpoint2.pq",
    "queen_1": folder + "queen_1_checkpoint.pickle",
}
if all(os.path.exists(p) for p in _ckpt.values()):
    blg = gpd.read_parquet(_ckpt["blg"])
    tess = gpd.read_parquet(_ckpt["tess"])
    streets = gpd.read_parquet(_ckpt["streets"])
    with open(_ckpt["queen_1"], "rb") as f:
        queen_1 = pickle.load(f)
    print("Restored blg/tess/streets/queen_1 from checkpoint.")
else:
    blg.to_parquet(_ckpt["blg"])
    tess.to_parquet(_ckpt["tess"])
    streets.to_parquet(_ckpt["streets"])
    with open(_ckpt["queen_1"], "wb") as f:
        pickle.dump(queen_1, f)
    print("No checkpoint found; continuing in-memory and writing a fresh checkpoint.")

print(sorted(tess.columns))
print(sorted(blg.columns))
print(sorted(streets.columns))

In [ ]:
print(type(queen_1))


In [ ]:
# restart from here after queen1 checkpoint

%time streets["sdsLen"] = momepy.Perimeter(streets).series
%time tess["stcSAl"] = momepy.StreetAlignment(tess, streets, "stcOri", "edgeID_primary").series
%time blg["stbSAl"] = momepy.StreetAlignment(blg, streets, "stbOri", "edgeID_primary").series


In [ ]:
%time profile = momepy.StreetProfile(streets, blg, distance=3)
streets["sdsSPW"] = profile.w
streets["sdsSPO"] = profile.o
streets["sdsSWD"] = profile.wd


In [ ]:
%time streets["sssLin"] = momepy.Linearity(streets).series

In [ ]:
%%time
# Area covered by each edge
vals = {x: [] for x in range(len(streets))}
for i, keys in enumerate(tess.edgeID_keys):
    for k in keys:
        vals[k].append(i)
area_sums = []
for inds in vals.values():
    area_sums.append(tess.sdcAre.iloc[inds].sum())
streets['sdsAre'] = area_sums


In [ ]:
%%time
# Buildings per meter
bpm = []
for inds, l in zip(vals.values(), streets.sdsLen):
    bpm.append(tess.buildings.iloc[inds].notna().sum() / l if len(inds) > 0 else 0)
streets['sisBpM'] = bpm


The street network forms a graph. We can use this property to first define a context of each segment.

In [ ]:
streets = streets.reset_index(drop=True)

str_q1 = libpysal.weights.contiguity.Queen.from_dataframe(streets, silence_warnings=True)

streets["misRea"] = momepy.Reached(
    streets, tess, "edgeID_primary", "edgeID_primary", spatial_weights=str_q1, mode="count"
).series
streets["mdsAre"] = momepy.Reached(streets, tess, "edgeID_primary", "edgeID_primary", spatial_weights=str_q1,
                               mode="sum").series


Then we create a graph representation and measure connectivity characters. The cell below creates a `networkx.MultiGraph`, measures characters and returns two GeoDataFrames &mdash; original segments and nodes &mdash; accompanied by spatial weights.

In [ ]:
%time graph = momepy.gdf_to_nx(streets)

print("node degree")
graph = momepy.node_degree(graph)

print("subgraph")
graph = momepy.subgraph(
    graph,
    radius=5,
    meshedness=True,
    cds_length=False,
    mode="sum",
    degree="degree",
    length="mm_len",
    mean_node_degree=False,
    proportion={0: True, 3: True, 4: True},
    cyclomatic=False,
    edge_node_ratio=False,
    gamma=False,
    local_closeness=True,
    closeness_weight="mm_len",
)
print("cds length")
graph = momepy.cds_length(graph, radius=3, name="ldsCDL")

print("clustering")
graph = momepy.clustering(graph, name="xcnSCl")

print("mean_node_dist")
graph = momepy.mean_node_dist(graph, name="mtdMDi")

#### Street network importance (betweenness centrality)

This is a whole-network measure of how much through-movement passes over each street (computed once here on the same `networkx` graph momepy already built above), conceptually different from the *local*, radius-limited characters above (`meshedness`, `local_closeness`, `degree`, ...), which describe built network form and stay there. This one describes the street's functional role for movement across the whole city, so it's carried into the **function** dataset in Section 2, not into `characters`/`data.pq`.

In [ ]:
%%time
# Exact betweenness centrality is O(V*E); fall back to a fixed-seed sample on
# very large graphs to keep runtime reasonable.
_n_nodes = graph.number_of_nodes()
_k = None if _n_nodes <= 8000 else 1000
print(f"Computing betweenness centrality on {_n_nodes} nodes (k={_k})...")
betweenness = nx.betweenness_centrality(graph, weight="mm_len", normalized=True, k=_k, seed=42 if _k else None)
nx.set_node_attributes(graph, betweenness, "betweenness")

In [ ]:
%time nodes, edges, sw = momepy.nx_to_gdf(graph, spatial_weights=True)

Weights and connections allow us to measure characters on a "subgraph" even without the networkX object.

In [ ]:
%time edges_w3 = momepy.sw_high(k=3, gdf=edges)
%time edges["ldsMSL"] = momepy.SegmentsLength(edges, spatial_weights=edges_w3, mean=True).series

%time nodes_w5 = momepy.sw_high(k=5, weights=sw)
%time nodes["lddNDe"] = momepy.NodeDensity(nodes, edges, nodes_w5).series
nodes["linWID"] = momepy.NodeDensity(
    nodes, edges, nodes_w5, weighted=True, node_degree="degree"
).series


Now we also include enclosures in the set.

In [ ]:
enclosures["ldeAre"] = momepy.Area(enclosures).series
enclosures["ldePer"] = momepy.Perimeter(enclosures).series
enclosures["lseCCo"] = momepy.CircularCompactness(enclosures, "ldeAre").series
enclosures["lseERI"] = momepy.EquivalentRectangularIndex(enclosures, "ldeAre", "ldePer").series
enclosures["lseCWA"] = momepy.CompactnessWeightedAxis(enclosures, "ldeAre", "ldePer").series
enclosures["lteOri"] = momepy.Orientation(enclosures).series

blo_q1 = libpysal.weights.contiguity.Queen.from_dataframe(enclosures, ids="eID")

inp, res = enclosures.sindex.query_bulk(enclosures.geometry, predicate='intersects')
indices, counts = np.unique(inp, return_counts=True)
enclosures['neighbors'] = counts - 1
enclosures['lteWNB'] = enclosures['neighbors'] / enclosures['ldePer']


In [ ]:
# Measure weighted cells within enclosure
encl_counts = tess.groupby('eID').count()
merged = enclosures[['eID', 'ldeAre']].merge(encl_counts[['tessellation']], how='left', on='eID')
enclosures['lieWCe'] = merged['tessellation'] / merged['ldeAre']


In [ ]:
tess['ltcWRE'] = momepy.BlocksCount(tess, 'eID', queen_3, 'tID').series


We will need to link data on nodes to tessellation, so we need to get the nearest network-based node ID.

In [ ]:
# get node id
%time links = momepy.get_network_ratio(tess, edges)
tess[['edgeID_keys2', 'edgeID_values2']] = links
%time tess['nodeID'] = momepy.get_node_id(tess, nodes, edges, node_id='nodeID', edge_keys='edgeID_keys2', edge_values='edgeID_values2')


In [ ]:
%%time
nodes["sddAre"] = momepy.Reached(
    nodes, tess, "nodeID", "nodeID", mode="sum", values="sdcAre"
).series


In [ ]:
tess.to_parquet(folder + "tess.pq")
blg.to_parquet(folder + "blg.pq")
nodes.to_parquet(folder + "nodes.pq")
edges.to_parquet(folder + "edges.pq")
enclosures.to_parquet(folder + "enclosures.pq")


Finally, we link all GeoDataFrames together.

In [ ]:
data = tess.merge(
    blg[
        [
            "tID",
            "sdbAre",
            "sdbPer",
            "sdbCoA",
            "ssbCCo",
            "ssbCor",
            "ssbSqu",
            "ssbERI",
            "ssbElo",
            "ssbCCM",
            "ssbCCD",
            "mtbSWR",
            "ldbPWL",
            "stbSAl",
            "libNCo",
        ]
    ],
    on="tID",
    how="left",
)
data = data.merge(
    edges[
        [
            "sdsLen",
            "sdsSPW",
            "sdsSPO",
            "sdsSWD",
            "sssLin",
            "sdsAre",
            "sisBpM",
            "misRea",
            "mdsAre",
            "ldsMSL",
            "edgeID_primary",
        ]
    ],
    on="edgeID_primary",
    how="left",
)
data = data.merge(
    nodes[
        [
            "degree",
            "meshedness",
            "proportion_3",
            "proportion_4",
            "proportion_0",
            "local_closeness",
            "ldsCDL",
            "xcnSCl",
            "mtdMDi",
            "nodeID",
            "lddNDe",
            "linWID",
            "sddAre",
        ]
    ],
    on="nodeID",
    how="left",
)
data = data.merge(
    enclosures[
        [
            "eID",
            "ldeAre",
            "ldePer",
            "lseCCo",
            "lseERI",
            "lseCWA",
            "lteOri",
            "lteWNB",
            "lieWCe",
        ]
    ],
    on="eID",
    how="left",
)


In [ ]:
data.columns


Now we specify columns representing characters and nothing else (geometry, IDs).

In [ ]:
characters = [
    "stcOri",
    "stbOri",
    "stbCeA",
    "sdcLAL",
    "sdcAre",
    "sscCCo",
    "sscERI",
    "sicCAR",
    "mtbAli",
    "mtbNDi",
    "mtcWNe",
    "mdcAre",
    "ltbIBD",
    "stcSAl",
    "ltcWRE",
    "sdbAre",
    "sdbPer",
    "sdbCoA",
    "ssbCCo",
    "ssbCor",
    "ssbSqu",
    "ssbERI",
    "ssbElo",
    "ssbCCM",
    "ssbCCD",
    "mtbSWR",
    "ldbPWL",
    "stbSAl",
    "libNCo",
    "sdsLen",
    "sdsSPW",
    "sdsSPO",
    "sdsSWD",
    "sssLin",
    "sdsAre",
    "sisBpM",
    "misRea",
    "mdsAre",
    "ldsMSL",
    "degree",
    "meshedness",
    "proportion_3",
    "proportion_4",
    "proportion_0",
    "local_closeness",
    "ldsCDL",
    "xcnSCl",
    "mtdMDi",
    "lddNDe",
    "linWID",
    "sddAre",
    "ldeAre",
    "ldePer",
    "lseCCo",
    "lseERI",
    "lseCWA",
    "lteOri",
    "lteWNB",
    "lieWCe",
]


Now we save all data to a parquet file.

In [ ]:
data[characters + ['tID', 'uID', 'tessellation', 'buildings']].to_parquet(folder + "data.pq")


#### 1.4.2. Contextualisation

Cluster analysis needs to be performant on large datasets, so it should not contain spatial constraints (such algorithms do not scale well). At the same time, we are interested in the tendency of characters within an area. Most primary characters are local only, with minimal sense of context. We therefore contextualise each primary character to get the first, second and third quartile of the distribution of values within an inclusive 10th order of contiguity around each tessellation cell, weighted by the inverse distance between cell centroids.

First we load the data (unless it's still in memory from the steps above).

In [ ]:
data = gpd.read_parquet(folder + "data.pq")


We prepare `gdf`, which contains characters and tessellation centroids as geometry, and `W`, a spatial weights matrix representing the 10th order of contiguity (inclusive).

In [ ]:
gdf = gpd.GeoDataFrame(data[characters], geometry=data.tessellation.centroid)
%time w1 = libpysal.weights.Queen.from_dataframe(data, geom_col='tessellation')
A = w1.sparse.astype(bool)  # 1st-order adjacency only; k=10 neighbours are computed in batches below

Then we loop over the GeoDataFrame and measure the contextualisation of each character. (Note: `.iteritems()` was removed from pandas in 2.0; we use `.items()` instead.)

In [ ]:
import scipy.sparse as sp

N = A.shape[0]
K = 10
BATCH_SIZE = 2000  # tune down if memory is tight, up if there is headroom

convolutions = {c: [None] * N for c in characters}

# measure convolutions, processing cells in batches so the full k=10
# neighbourhood matrix (which caused OOM) is never held in memory at once.
# Mathematically identical to momepy.sw_high(k=10, ...): for each batch of
# source cells we accumulate, via repeated sparse matrix multiplication,
# every cell reachable by a walk of length 2..10 in the Queen adjacency
# graph (excluding the cell itself) - same definition momepy uses internally.
for start in tqdm(range(0, N, BATCH_SIZE), total=-(-N // BATCH_SIZE)):
    end = min(start + BATCH_SIZE, N)
    idx = np.arange(start, end)
    b = end - start

    S = sp.csr_matrix((np.ones(b, dtype=bool), (np.arange(b), idx)), shape=(b, N))
    walk = S @ A
    acc = sp.csr_matrix((b, N), dtype=bool)
    for _ in range(2, K + 1):
        walk = (walk.astype(int) @ A.astype(int)) > 0
        acc = acc + walk
    acc = acc.tolil()
    for local_i, global_i in enumerate(idx):
        acc[local_i, global_i] = False  # exclude self, matching momepy.sw_high
    acc = acc.tocsr()

    for local_i, global_i in enumerate(idx):
        neighbours = acc.indices[acc.indptr[local_i]:acc.indptr[local_i + 1]]
        geom = gdf.geometry.iloc[global_i]
        vicinity = gdf.iloc[neighbours]
        distance = vicinity.distance(geom)
        distance_decay = (1 / distance).replace(np.inf, 0)  # to filter out duplications

        for c in characters:
            values = vicinity[c].values
            sorter = np.argsort(values)
            values = values[sorter]
            nan_mask = np.isnan(values)
            if nan_mask.all():
                convolutions[c][global_i] = np.array([np.nan] * 3)
            else:
                sample_weight = distance_decay.values[sorter][~nan_mask]
                weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
                weighted_quantiles /= np.sum(sample_weight)
                interpolate = np.interp([.25, .5, .75], weighted_quantiles, values[~nan_mask])
                convolutions[c][global_i] = interpolate

The resulting dictionary can then be exploded into a DataFrame with the tessellation ID as an index.

In [ ]:
%time conv = pd.DataFrame(convolutions, index=data.index)
%time exploded = pd.concat([pd.DataFrame(conv[c].to_list(), columns=[c + '_q1', c + '_q2', c + '_q3']) for c in characters], axis=1)


In [ ]:
exploded.index = data.tID
exploded


And saved to a file, together with the weights matrix.

In [ ]:
exploded.to_parquet(folder + "convolutions.pq")


Visualize all of the results we have until now, before going into further analysis  

In [ ]:
import lonboard
from lonboard.colormap import apply_continuous_cmap
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import ipywidgets as widgets
from IPython.display import display

# --- mappa codice -> nome leggibile (presa dalla cella 249 del notebook) ---
char_names = {
    'sdbAre': 'area of building', 'sdbPer': 'perimeter of building',
    'sdbCoA': 'courtyard area of building', 'ssbCCo': 'circular compactness of building',
    'ssbCor': 'corners of building', 'ssbSqu': 'squareness of building',
    'ssbERI': 'equivalent rectangular index of building', 'ssbElo': 'elongation of building',
    'ssbCCM': 'centroid - corner mean distance of building',
    'ssbCCD': 'centroid - corner distance deviation of building',
    'stbOri': 'orientation of building', 'sdcLAL': 'longest axis length of ETC',
    'sdcAre': 'area of ETC', 'sscCCo': 'circular compactness of ETC',
    'sscERI': 'equivalent rectangular index of ETC', 'stcOri': 'orientation of ETC',
    'sicCAR': 'covered area ratio of ETC', 'stbCeA': 'cell alignment of building',
    'mtbAli': 'alignment of neighbouring buildings',
    'mtbNDi': 'mean distance between neighbouring buildings',
    'mtcWNe': 'perimeter-weighted neighbours of ETC',
    'mdcAre': 'area covered by neighbouring cells',
    'ltcWRE': 'weighted reached enclosures of ETC', 'ltbIBD': 'mean inter-building distance',
    'sdsSPW': 'width of street profile', 'sdsSWD': 'width deviation of street profile',
    'sdsSPO': 'openness of street profile', 'sdsLen': 'length of street segment',
    'sssLin': 'linearity of street segment', 'ldsMSL': 'mean segment length within 3 steps',
    'degree': 'node degree of junction', 'meshedness': 'local meshedness of street network',
    'proportion_3': 'local proportion of 3-way intersections',
    'proportion_4': 'local proportion of 4-way intersections',
    'proportion_0': 'local proportion of cul-de-sacs',
    'local_closeness': 'local closeness of street network',
    'ldsCDL': 'local cul-de-sac length', 'xcnSCl': 'square clustering of street network',
    'mtdMDi': 'mean distance to neighbouring nodes',
    'lddNDe': 'local node density of street network',
    'linWID': 'local degree weighted node density',
    'stbSAl': 'street alignment of building', 'stcSAl': 'street alignment of ETC',
    'mtbSWR': 'shared walls ratio of buildings',
    'sddAre': 'area covered by node-attached ETCs',
    'sdsAre': 'area covered by edge-attached ETCs',
    'sisBpM': 'buildings per meter of street segment',
    'mdsAre': 'reached area by neighbouring segments',
    'ldeAre': 'area of enclosure', 'ldePer': 'perimeter of enclosure',
    'lseCCo': 'circular compactness of enclosure',
    'lseERI': 'equivalent rectangular index of enclosure',
    'lseCWA': 'compactness-weighted axis of enclosure', 'lteOri': 'orientation of enclosure',
    'lteWNB': 'perimeter-weighted neighbours of enclosure',
    'lieWCe': 'area-weighted ETCs of enclosure',
    'ldbPWL': 'perimeter wall length of adjacent buildings',
    'libNCo': 'number of courtyards within adjacent buildings',
    'misRea': 'cells reached within neighbouring street segments',
}
quartile_labels = {'q1': '1° quartile', 'q2': 'mediana', 'q3': '3° quartile'}

def context_label(col):
    base, q = col.rsplit('_', 1)
    return f"{char_names.get(base, base)} — {quartile_labels.get(q, q)} (contesto 10° ordine)"

# Legge solo geometria e ID. Il file completo delle convoluzioni ha 178 colonne
# e il merge di tutte le colonne richiede diversi GB senza essere necessario.
tess_path = folder + "tess.pq"
conv_path = folder + "convolutions.pq"
print("Caricamento delle geometrie della mappa...")
tess_native = gpd.read_parquet(tess_path, columns=['tID', 'tessellation']).set_geometry('tessellation')
tess_ids = tess_native['tID'].to_numpy()
# 339k celle con poligoni dettagliati (fino a migliaia di vertici ciascuna) mandano in
# crash il rendering come PolygonLayer (Arrow table da decine di GB). Usiamo i centroidi,
# calcolati nel CRS originale (piu' corretto e piu' veloce che riproiettare i poligoni):
# stessa scala colori interattiva, payload ~5MB invece di ~25GB.
centroids = gpd.GeoDataFrame(geometry=tess_native.geometry.centroid, crs=tess_native.crs).to_crs(4326)

exclude = {'uID', 'eID', 'tID', 'edgeID_primary', 'nodeID'}
tess_schema = pq.read_schema(tess_path)
raw_cols = [
    field.name for field in tess_schema
    if (pa.types.is_integer(field.type) or pa.types.is_floating(field.type))
    and field.name not in exclude
]
context_cols = [
    name for name in pq.read_schema(conv_path).names
    if name.rsplit('_', 1)[-1] in quartile_labels
]

dropdown_options = (
    [(char_names.get(c, c), c) for c in raw_cols]
    + [(context_label(c), c) for c in context_cols]
)

def load_values(column):
    if column in context_cols:
        values = pd.read_parquet(conv_path, columns=[column])[column]
        return values.reindex(tess_ids).to_numpy()
    return pd.read_parquet(tess_path, columns=[column])[column].to_numpy()

def get_colors(column):
    values = load_values(column)
    vmin, vmax = np.nanmin(values), np.nanmax(values)
    normalized = np.zeros_like(values, dtype=float) if vmin == vmax else (values - vmin) / (vmax - vmin)
    return apply_continuous_cmap(normalized, cm.get_cmap('viridis')), vmin, vmax

def legend_html(column, vmin, vmax, cmap_name='viridis'):
    cmap = cm.get_cmap(cmap_name)
    stops = ','.join(
        f'rgb({int(r*255)},{int(g*255)},{int(b*255)}) {i/9*100:.0f}%'
        for i, (r, g, b, _) in enumerate(cmap(np.linspace(0, 1, 10)))
    )
    label = char_names.get(column, context_label(column) if column in context_cols else column)
    return f'''
    <div style="font-family:sans-serif;padding:6px 0">
      <b>{label}</b> <span style="color:#888;font-size:11px">({column})</span><br>
      <div style="width:280px;height:18px;background:linear-gradient(to right,{stops});
                  border:1px solid #333;border-radius:2px"></div>
      <div style="display:flex;justify-content:space-between;width:280px;font-size:12px">
        <span>{vmin:.2f}</span><span>{vmax:.2f}</span>
      </div>
    </div>
    '''

dropdown = widgets.Dropdown(options=dropdown_options, value='sdcAre', description='Variabile:')
legend_out = widgets.Output()

initial_colors, vmin, vmax = get_colors(dropdown.value)
layer = lonboard.ScatterplotLayer.from_geopandas(
    centroids,
    get_fill_color=initial_colors,
    radius_min_pixels=1.5,
    radius_max_pixels=4,
)
m = lonboard.Map(layer)

def refresh_legend(column, vmin, vmax):
    legend_out.clear_output()
    with legend_out:
        display(widgets.HTML(legend_html(column, vmin, vmax)))

refresh_legend(dropdown.value, vmin, vmax)

def on_change(change):
    column = change['new']
    colors, vmin, vmax = get_colors(column)
    layer.get_fill_color = colors
    refresh_legend(column, vmin, vmax)

dropdown.observe(on_change, names='value')

display(widgets.VBox([dropdown, legend_out, m]))


## 2. Analysis of function

Next we want to characterise the built environment by its function. Singapore's function layer draws on several `data.gov.sg` datasets that have no single Winterthur-wide equivalent, so **every layer below is sourced from OpenStreetMap** instead, keeping the notebook to one external data provider for this whole section.

### 2.1. Eating establishments

We query OSM for common eating-related `amenity` tags. Points (and polygons, whose centroid we take) are linked to tessellation based on accessibility: as in the Singapore notebook, we use `pandana` to measure the number of POIs within a 15 minute walk (represented as 1200 m on the street network).

In [ ]:
tess = gpd.read_parquet(folder + "tess.pq", columns=["tessellation", 'tID'])
edges = gpd.read_parquet(folder + "edges.pq")
nodes = gpd.read_parquet(folder + "nodes.pq")


We build the pandana network from our existing nodes and edges data.

In [ ]:
nodes = nodes.set_index('nodeID')
network = pandana.Network(nodes.geometry.x, nodes.geometry.y,
                          edges['node_start'], edges['node_end'], edges[['mm_len']])


In [ ]:
%%time
eating_tags = {'amenity': ['restaurant', 'cafe', 'fast_food', 'bar', 'pub', 'biergarten', 'food_court']}
eating = ox.geometries_from_place(place, eating_tags)
eating = eating.to_crs(nodes.crs)
eating['geometry'] = eating.geometry.centroid


In [ ]:
%%time
network.set_pois(category='eating',
                 maxdist=1200,
                 maxitems=eating.shape[0],
                 x_col=eating.geometry.x,
                 y_col=eating.geometry.y)


And now we measure the number of points within 1200 m around each node. Notice `count(axis=1)` at the end of the cell.

In [ ]:
%%time
nodes['eating'] = network.nearest_pois(distance=1200,
                               category='eating',
                               num_pois=eating.shape[0],
                               include_poi_ids=False).replace(1200, pd.NA).count(axis=1)


### 2.2 Supermarkets

Supermarkets are tagged `shop=supermarket` in OSM.

In [ ]:
%%time
supermarket_tags = {'shop': 'supermarket'}
supermarkets = ox.geometries_from_place(place, supermarket_tags)
supermarkets = supermarkets.to_crs(nodes.crs)
supermarkets['geometry'] = supermarkets.geometry.centroid


We can directly link the locations to our existing pandana network as we did before.

In [ ]:
%%time
network.set_pois(category='supermarkets',
                 maxdist=1200,
                 maxitems=supermarkets.shape[0],
                 x_col=supermarkets.geometry.x,
                 y_col=supermarkets.geometry.y)


In [ ]:
%%time
nodes['supermarkets'] = network.nearest_pois(distance=1200,
                               category='supermarkets',
                               num_pois=supermarkets.shape[0],
                               include_poi_ids=False).replace(1200, pd.NA).count(axis=1)


### 2.3 Land Use

Land use is retrieved from OSM's `landuse` tag and kept at its full category granularity —
no collapsing into a handful of coarse buckets. Whatever distinct `landuse` values actually
occur in the city (residential, commercial, retail, industrial, farmland, forest, cemetery,
railway, ...) all become separate categories downstream.

In [ ]:
%%time
land_use = ox.geometries_from_place(place, tags={'landuse': True})
land_use = land_use[land_use.geom_type.isin(['Polygon', 'MultiPolygon'])]
land_use["use"] = land_use["landuse"]  # keep full OSM granularity, just alias the column


In [ ]:
land_use.landuse.value_counts()


Geometry needs validation, then we reproject to match the tessellation.

In [ ]:
land_use.geometry.is_valid.all()


In [ ]:
land_use.geometry = land_use.geometry.buffer(0)


In [ ]:
land_use = land_use.to_crs(tess.crs)


Now we join the land-use data to enclosed tessellation based on the largest intersection.

In [ ]:
%time tess = area_max(land_use, tess, 'use')


We fill any unjoined tessellation cells with `unknown`.

In [ ]:
tess.use = tess.use.fillna("unknown")

### 2.4 Parks

Parks were previously retrieved as OSM `leisure=park` features, which is inconsistently and
incompletely tagged — likely the source of past errors here. Instead we use Grün Stadt Zürich's
own green-space maintenance cadastre (WFS `Gruenflaechen`, CC0-licensed — this is the
authoritative source the city itself uses to manage these areas), categorised by `produkt`.
We treat `610 Parkanlagen` (actual managed parks, e.g. "Neumünsteranlage") as "parks" and
dissolve by named site (`pflegeareal`) to get coherent park polygons instead of thousands of
maintenance-unit slivers.

We also compute a second, broader measure — distance to the nearest public open space of *any*
kind (parks + cemeteries `620 Friedhöfe` + sports/pool grounds `630 Sport- und Badeanlagen`) —
since cemeteries and sports grounds function as accessible green/recreational space in a dense
city even though they aren't "parks" in the strict sense. `640 Weitere Freiräume` (traffic
islands, street-side planting strips, ...) is excluded from both as too fine-grained to mean
either "park" or meaningful "open space".

In [ ]:
path = folder + "gruenflaechen_osm.pq"
if os.path.exists(path):
    gruenflaechen = gpd.read_parquet(path)
else:
    r = requests.get(
        "https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Gruenflaechen",
        params={
            "SERVICE": "WFS",
            "VERSION": "1.1.0",
            "REQUEST": "GetFeature",
            "TYPENAME": "gruenflaechen",
            "OUTPUTFORMAT": "application/vnd.geo+json",
        },
        headers={"User-Agent": "spatial-signatures-zurich-city (research script)"},
        timeout=300,
    )
    r.raise_for_status()
    gruenflaechen = gpd.GeoDataFrame.from_features(r.json()["features"], crs="EPSG:4326")
    gruenflaechen = gruenflaechen[["produkt", "pflegeareal", "geometry"]]
    gruenflaechen.to_parquet(path)
gruenflaechen = gruenflaechen.to_crs(tess.crs)
gruenflaechen["geometry"] = gruenflaechen.geometry.make_valid()  # real-world cadastre data; dissolve() below needs valid geometry
gruenflaechen.produkt.value_counts()

In [ ]:
parks = (
    gruenflaechen[gruenflaechen.produkt == "610 Parkanlagen"]
    .dissolve(by="pflegeareal")
    .reset_index()
)
open_space = gruenflaechen[
    gruenflaechen.produkt.isin(["610 Parkanlagen", "620 Friedhöfe", "630 Sport- und Badeanlagen"])
]
len(parks), len(open_space)

Distance from each cell to the nearest park, and separately to the nearest open space of any kind, same expanding-buffer `sjoin_nearest` approach as before.

In [ ]:
tess_geom = gpd.GeoDataFrame(geometry=tess.tessellation, crs=tess.crs)

nearest_park = tess_geom.sjoin_nearest(parks[["geometry"]], distance_col="park_distance")
tess["park_distance"] = nearest_park["park_distance"].groupby(level=0).min()

nearest_open = tess_geom.sjoin_nearest(open_space[["geometry"]], distance_col="open_space_distance")
tess["open_space_distance"] = nearest_open["open_space_distance"].groupby(level=0).min()

### 2.5 Historical monuments

Historical monuments are retrieved via OSM's `historic` tag (monuments, memorials, castles, etc.), linked to the pandana network exactly as eating establishments and supermarkets above.

In [ ]:
%%time
monument_tags = {'historic': ['monument', 'memorial', 'castle', 'fort', 'archaeological_site', 'heritage']}
monuments = ox.geometries_from_place(place, monument_tags).to_crs(tess.crs)
monuments['geometry'] = monuments.geometry.centroid


In [ ]:
network.set_pois(category='monuments',
                 maxdist=1200,
                 maxitems=monuments.shape[0],
                 x_col=monuments.geometry.x,
                 y_col=monuments.geometry.y)


In [ ]:
%%time
results = network.nearest_pois(distance=1200,
                               category='monuments',
                               num_pois=monuments.shape[0],
                               include_poi_ids=False).replace(1200, pd.NA).count(axis=1)


In [ ]:
nodes['monuments'] = results


### 2.6 Street network importance

Betweenness centrality was already computed in Section 1.4.1 on the same `networkx` graph momepy uses for the local street characters, since rebuilding the graph here would be wasteful. It's a whole-network measure of a street's importance for movement, not a built-form measurement, so it's linked in here through `nodes` (the same node-based link used for eating/supermarkets/monuments above) rather than carried in `data.pq`.

In [ ]:
nodeid = pd.read_parquet(folder + "tess.pq", columns=["nodeID"])


Finally, we link data attached to nodes to tessellation based on nodeID.

In [ ]:
tess['nodeID'] = nodeid.nodeID.values


In [ ]:
tess = tess.merge(nodes[['eating', 'supermarkets', 'monuments', 'betweenness']], on='nodeID', how='left')

### 2.7 Population

We use the Canton of Zürich's official, register-based hectare-grid population statistics
(`stat_bevoelkerung_ha`, CC0-licensed, published via the cantonal Open Government Data WFS,
current to 2025-12-31) instead of the WorldPop global raster used in the canton notebook — a
real, current, high-resolution count rather than a modeled/dasymetric estimate.

One important data-quality wrinkle, found by testing this live before trusting it: the source
suppresses small counts as `-999` for privacy (when the true value is between 1 and 3). Naively
summing that column subtracts population instead of just losing it — verified live, this alone
put a bbox-wide total ~330k people too *low*. We replace suppressed counts with 2 (roughly the
midpoint of the 1-3 range they represent) before interpolating.

Population (and the other hectare-grid layers below) is interpolated directly onto tessellation
cells rather than building footprints: cells tile all buildable land with no gaps, so no
population mass is lost to the courtyards/private gardens/set-backs between a building and its
cell boundary the way it would interpolating onto bare building footprints. Mass genuinely
outside any enclosure (water, parks, ...) is correctly excluded since those have no cells.

In [ ]:
KANTON_WFS = "https://maps.zh.ch/wfs/OGDZHWFS"


def fetch_wfs_geojson(url, params, crs, timeout=180):
    headers = {"User-Agent": "spatial-signatures-zurich-city (research script)"}
    r = requests.get(url, params=params, timeout=timeout, headers=headers)
    r.raise_for_status()
    return gpd.GeoDataFrame.from_features(r.json()["features"], crs=crs)


def fetch_kanton_layer(typename, path, bbox):
    """Fetch (or load a cached copy of) a Kanton Zürich OGD WFS layer, bbox-restricted
    to the study area. -999 (privacy-suppressed) values become NaN here; callers decide
    how to fill them back in depending on whether a column is extensive (a count, which
    should not silently lose mass) or intensive (a share/percentage, safe to leave as NaN
    and fill with 0 later, same as the rest of the function data)."""
    if os.path.exists(path):
        return gpd.read_parquet(path)
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typeName": typename,
        "outputFormat": "application/json; subtype=geojson",
        "bbox": bbox,
    }
    gdf = fetch_wfs_geojson(KANTON_WFS, params, crs="EPSG:2056")
    gdf = gdf.replace(-999.0, np.nan)
    gdf.to_parquet(path)
    return gdf


_admin_2056 = gpd.read_parquet(folder + "admin_osm.pq").to_crs(2056)
_minx, _miny, _maxx, _maxy = _admin_2056.total_bounds
KANTON_BBOX = f"{_minx},{_miny},{_maxx},{_maxy},urn:ogc:def:crs:EPSG::2056"

In [ ]:
pop_ha = fetch_kanton_layer(
    "ms:ogd-0063_stat_bevoelkerung_ha_f", folder + "bevoelkerung_ha.pq", KANTON_BBOX
).to_crs(tess.crs)

pop_ha["pers_n"] = pop_ha["pers_n"].fillna(2)  # -999 -> midpoint of the suppressed 1-3 range
intensive_pop = ["frau_p", "ausland_p", "altersquot", "jugendquot", "gesamtquot"]

tess = area_interpolate(pop_ha, tess, "pers_n")
tess = area_max(pop_ha, tess, intensive_pop)
tess = tess.rename(columns={"pers_n": "population"})

In [ ]:
print(f"Interpolated population total: {tess.population.sum():,.0f}")
assert 300_000 < tess.population.sum() < 600_000, (
    "Interpolated city population is well outside the plausible range for Zürich (~450k) "
    "-- check the WFS fetch and the -999 handling above."
)

### 2.8 Employment

Job counts and sector mix from the Canton's official hectare-grid employment statistics (`stat_beschaeftigte_ha`) — the direct analogue of the economic-census layer already used for Barcelona in this repository, just sourced officially instead of scraped/derived.

In [ ]:
emp_ha = fetch_kanton_layer(
    "ms:ogd-0142_stat_beschaeftigte_ha_f", folder + "beschaeftigte_ha.pq", KANTON_BBOX
).to_crs(tess.crs)

extensive_emp = ["betr_n", "besch_n", "vzae_n"]  # establishments, employees, full-time-equivalents
emp_ha[extensive_emp] = emp_ha[extensive_emp].fillna(2)
intensive_emp = ["besch_f_p", "hightech_p", "wissdl_p"] + sorted(
    c for c in emp_ha.columns if c.startswith("noga08_")
)  # % female, % high-tech, % knowledge-intensive services, % by NOGA08 sector

tess = area_interpolate(emp_ha, tess, extensive_emp)
tess = area_max(emp_ha, tess, intensive_emp)
tess = tess.rename(columns={"betr_n": "jobs_establishments", "besch_n": "jobs_employed", "vzae_n": "jobs_fte"})

### 2.9 Building stock

Building-age mix, building-type mix, floor-count mix, and dwelling counts/living area from the Canton's official hectare-grid building statistics (`stat_gebaeude_ha`). This is statistical/usage information about the existing building stock, not a geometric shape measurement, so it belongs here in function rather than in the form characters.

In [ ]:
bld_ha = fetch_kanton_layer(
    "ms:ogd-0272_stat_gebaeude_ha_f", folder + "gebaeude_ha.pq", KANTON_BBOX
).to_crs(tess.crs)

extensive_bld = ["geb_n", "whg_n", "wfl_qm"]  # buildings, dwellings, living area (sqm)
bld_ha[extensive_bld] = bld_ha[extensive_bld].fillna(2)
intensive_bld = [
    "bj_v45_p", "bj_46_80_p", "bj_81_00_p", "bj_n01_p",  # building-age mix
    "efh_p", "mfh_p", "mischn_p", "anderen_p",  # building-type mix
    "stwk_1_2_p", "stwk_3_4_p", "stwk_5_6_p", "stwk_ab7_p",  # floor-count mix
]

tess = area_interpolate(bld_ha, tess, extensive_bld)
tess = area_max(bld_ha, tess, intensive_bld)
tess = tess.rename(columns={"geb_n": "buildings_n", "whg_n": "dwellings_n", "wfl_qm": "living_area_sqm"})

### 2.10 Public transport quality & planning typology

The Canton publishes each resident's public-transport quality class (`afv_gv_ew_oevgk`) — an official accessibility classification based on distance-weighted service frequency to the nearest stops — together with a spatial-planning typology (urban/non-urban, and a more detailed category such as "Stadtlandschaft"/"Landschaft unter Druck"). This is point data (one point per aggregated residential location), so we link it by nearest point rather than by area overlap.

In [ ]:
oevgk = fetch_kanton_layer(
    "ms:ogd-0412_afv_gv_ew_oevgk_p", folder + "oevgk_ew.pq", KANTON_BBOX
).to_crs(tess.crs)

tess_geom = gpd.GeoDataFrame(geometry=tess.tessellation, crs=tess.crs)
matched = tess_geom.sjoin_nearest(
    oevgk[["gk_kanton", "urbanitaet", "rok_bez", "geometry"]], how="left", distance_col="oevgk_dist"
)
matched = matched[~matched.index.duplicated(keep="first")]

tess["transit_quality"] = matched["gk_kanton"]
tess["urbanity"] = matched["urbanitaet"]
tess["planning_typology"] = matched["rok_bez"]

### 2.11 NDVI &mdash; TODO

The Singapore notebook computes NDVI from a Sentinel-2 composite mosaic, using a local tile index (`../../data_processing/imagery/GHS-composite-S2.geojson`) that isn't part of this repository, so the tiling logic can't be carried over as-is.

To implement this for Winterthur, the most direct modern replacement would be to query Sentinel-2 L2A scenes for the municipal bounding box from a public STAC catalog (e.g. Microsoft Planetary Computer or Element84's Earth Search), composite/clip to `tess.total_bounds`, compute `(NIR - Red) / (NIR + Red)`, then run the same `rasterstats.zonal_stats(..., stats=['mean'])` pattern used for population above. That requires two extra packages (`pystac-client`, `planetary-computer`) not currently installed, since this step is left unimplemented for now.

In [ ]:
# TODO: implement NDVI for the City of Zürich.
# Suggested approach: fetch Sentinel-2 L2A imagery for tess.total_bounds from a public
# STAC catalog (e.g. Microsoft Planetary Computer), compute NDVI = (B08 - B04) / (B08 + B04),
# then zonal_stats(tess.geometry, ndvi_array, affine=..., stats=['mean']) as done for population.
# tess['ndvi'] = ...


### 2.12. Night lights &mdash; TODO

The Singapore notebook's night-lights step requires a manual, login-gated download from https://eogdata.mines.edu/download_dnb_composites.html (VIIRS DNB monthly composite), and even there the tile ID (`75N060E`) is specific to Singapore's location. The equivalent Winterthur tile would need to be identified and downloaded by hand from the same portal (an EOG/NASA Earthdata login is required; this cannot be automated from an unattended notebook cell), then processed with the same `rio.clip_box` / `rio.reproject` / `zonal_stats` pattern used in the Singapore notebook.

In [ ]:
# TODO: implement night lights for the City of Zürich.
# 1. Manually download the VIIRS DNB monthly composite tile covering Zurich from
#    https://eogdata.mines.edu/download_dnb_composites.html (requires login).
# 2. nl = rioxarray.open_rasterio(folder + '<downloaded_tile>.tif')
# 3. Clip to tess.total_bounds, reproject to tess.crs, and run rasterstats.zonal_stats
#    exactly as done for population/NDVI above.
# tess['lights'] = ...


In [ ]:
tess.drop(columns=["tessellation", 'nodeID']).to_parquet(folder + "func_data_tess.pq")


In [ ]:
import lonboard
from lonboard.colormap import apply_continuous_cmap
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import ipywidgets as widgets
from IPython.display import display

# --- mappa codice -> nome leggibile (presa dalla cella 249 del notebook) ---
char_names = {
    'sdbAre': 'area of building', 'sdbPer': 'perimeter of building',
    'sdbCoA': 'courtyard area of building', 'ssbCCo': 'circular compactness of building',
    'ssbCor': 'corners of building', 'ssbSqu': 'squareness of building',
    'ssbERI': 'equivalent rectangular index of building', 'ssbElo': 'elongation of building',
    'ssbCCM': 'centroid - corner mean distance of building',
    'ssbCCD': 'centroid - corner distance deviation of building',
    'stbOri': 'orientation of building', 'sdcLAL': 'longest axis length of ETC',
    'sdcAre': 'area of ETC', 'sscCCo': 'circular compactness of ETC',
    'sscERI': 'equivalent rectangular index of ETC', 'stcOri': 'orientation of ETC',
    'sicCAR': 'covered area ratio of ETC', 'stbCeA': 'cell alignment of building',
    'mtbAli': 'alignment of neighbouring buildings',
    'mtbNDi': 'mean distance between neighbouring buildings',
    'mtcWNe': 'perimeter-weighted neighbours of ETC',
    'mdcAre': 'area covered by neighbouring cells',
    'ltcWRE': 'weighted reached enclosures of ETC', 'ltbIBD': 'mean inter-building distance',
    'sdsSPW': 'width of street profile', 'sdsSWD': 'width deviation of street profile',
    'sdsSPO': 'openness of street profile', 'sdsLen': 'length of street segment',
    'sssLin': 'linearity of street segment', 'ldsMSL': 'mean segment length within 3 steps',
    'degree': 'node degree of junction', 'meshedness': 'local meshedness of street network',
    'proportion_3': 'local proportion of 3-way intersections',
    'proportion_4': 'local proportion of 4-way intersections',
    'proportion_0': 'local proportion of cul-de-sacs',
    'local_closeness': 'local closeness of street network',
    'ldsCDL': 'local cul-de-sac length', 'xcnSCl': 'square clustering of street network',
    'mtdMDi': 'mean distance to neighbouring nodes',
    'lddNDe': 'local node density of street network',
    'linWID': 'local degree weighted node density',
    'stbSAl': 'street alignment of building', 'stcSAl': 'street alignment of ETC',
    'mtbSWR': 'shared walls ratio of buildings',
    'sddAre': 'area covered by node-attached ETCs',
    'sdsAre': 'area covered by edge-attached ETCs',
    'sisBpM': 'buildings per meter of street segment',
    'mdsAre': 'reached area by neighbouring segments',
    'ldeAre': 'area of enclosure', 'ldePer': 'perimeter of enclosure',
    'lseCCo': 'circular compactness of enclosure',
    'lseERI': 'equivalent rectangular index of enclosure',
    'lseCWA': 'compactness-weighted axis of enclosure', 'lteOri': 'orientation of enclosure',
    'lteWNB': 'perimeter-weighted neighbours of enclosure',
    'lieWCe': 'area-weighted ETCs of enclosure',
    'ldbPWL': 'perimeter wall length of adjacent buildings',
    'libNCo': 'number of courtyards within adjacent buildings',
    'misRea': 'cells reached within neighbouring street segments',
    'park_distance': 'distance to nearest park',
    'open_space_distance': 'distance to nearest open/green space',
    'eating': 'eating establishments',
    'supermarkets': 'supermarkets',
    'monuments': 'historical monuments',
    'betweenness': 'street betweenness centrality',
    'population': 'population',
    'frau_p': '% female',
    'ausland_p': '% foreign origin',
    'altersquot': 'old-age dependency ratio',
    'jugendquot': 'youth dependency ratio',
    'gesamtquot': 'total dependency ratio',
    'jobs_establishments': 'business establishments',
    'jobs_employed': 'jobs',
    'jobs_fte': 'jobs (FTE)',
    'besch_f_p': '% female employees',
    'hightech_p': '% high-tech jobs',
    'wissdl_p': '% knowledge-intensive service jobs',
    'buildings_n': 'buildings',
    'dwellings_n': 'dwellings',
    'living_area_sqm': 'living area (sqm)',
    'bj_v45_p': '% buildings built before 1945',
    'bj_46_80_p': '% buildings built 1946-1980',
    'bj_81_00_p': '% buildings built 1981-2000',
    'bj_n01_p': '% buildings built after 2001',
    'efh_p': '% single-family houses',
    'mfh_p': '% multi-family houses',
    'mischn_p': '% mixed-use buildings',
    'anderen_p': '% other building types',
    'stwk_1_2_p': '% buildings with 1-2 floors',
    'stwk_3_4_p': '% buildings with 3-4 floors',
    'stwk_5_6_p': '% buildings with 5-6 floors',
    'stwk_ab7_p': '% buildings with 7+ floors',
}
quartile_labels = {'q1': '1\u00b0 quartile', 'q2': 'mediana', 'q3': '3\u00b0 quartile'}

def context_label(col):
    base, q = col.rsplit('_', 1)
    return f"{char_names.get(base, base)} \u2014 {quartile_labels.get(q, q)} (contesto 10\u00b0 ordine)"

# Checkpoint completo: geometria+form (tess.pq), form contestualizzato (convolutions.pq),
# function (func_data_tess.pq). Stessa logica di caricamento lazy della cella dopo la Part 1,
# cosi da poter controllare tutto (form + function) prima di lanciare il clustering.
tess_path = folder + "tess.pq"
conv_path = folder + "convolutions.pq"
func_path = folder + "func_data_tess.pq"

print("Caricamento delle geometrie della mappa...")
tess_native = gpd.read_parquet(tess_path, columns=['tID', 'tessellation']).set_geometry('tessellation')
tess_ids = tess_native['tID'].to_numpy()
# 339k celle con poligoni dettagliati (fino a migliaia di vertici ciascuna) mandano in
# crash il rendering come PolygonLayer (Arrow table da decine di GB). Usiamo i centroidi,
# calcolati nel CRS originale (piu' corretto e piu' veloce che riproiettare i poligoni):
# stessa scala colori interattiva, payload ~5MB invece di ~25GB.
centroids = gpd.GeoDataFrame(geometry=tess_native.geometry.centroid, crs=tess_native.crs).to_crs(4326)

exclude = {'uID', 'eID', 'tID', 'edgeID_primary', 'nodeID'}
tess_schema = pq.read_schema(tess_path)
raw_cols = [
    field.name for field in tess_schema
    if (pa.types.is_integer(field.type) or pa.types.is_floating(field.type))
    and field.name not in exclude
]
context_cols = [
    name for name in pq.read_schema(conv_path).names
    if name.rsplit('_', 1)[-1] in quartile_labels
]
func_schema = pq.read_schema(func_path)
function_cols = [
    field.name for field in func_schema
    if (pa.types.is_integer(field.type) or pa.types.is_floating(field.type))
    and field.name not in exclude
]

dropdown_options = (
    [(char_names.get(c, c), c) for c in raw_cols]
    + [(context_label(c), c) for c in context_cols]
    + [(char_names.get(c, c) + ' [function]', c) for c in function_cols]
)

def load_values(column):
    if column in context_cols:
        values = pd.read_parquet(conv_path, columns=[column])[column]
        return values.reindex(tess_ids).to_numpy()
    if column in function_cols:
        values = pd.read_parquet(func_path, columns=['tID', column]).set_index('tID')[column]
        return values.reindex(tess_ids).to_numpy()
    return pd.read_parquet(tess_path, columns=[column])[column].to_numpy()

def get_colors(column):
    values = load_values(column)
    vmin, vmax = np.nanmin(values), np.nanmax(values)
    normalized = np.zeros_like(values, dtype=float) if vmin == vmax else (values - vmin) / (vmax - vmin)
    return apply_continuous_cmap(normalized, cm.get_cmap('viridis')), vmin, vmax

def legend_html(column, vmin, vmax, cmap_name='viridis'):
    cmap = cm.get_cmap(cmap_name)
    stops = ','.join(
        f'rgb({int(r*255)},{int(g*255)},{int(b*255)}) {i/9*100:.0f}%'
        for i, (r, g, b, _) in enumerate(cmap(np.linspace(0, 1, 10)))
    )
    label = char_names.get(column, context_label(column) if column in context_cols else column)
    return f'''
    <div style="font-family:sans-serif;padding:6px 0">
      <b>{label}</b> <span style="color:#888;font-size:11px">({column})</span><br>
      <div style="width:280px;height:18px;background:linear-gradient(to right,{stops});
                  border:1px solid #333;border-radius:2px"></div>
      <div style="display:flex;justify-content:space-between;width:280px;font-size:12px">
        <span>{vmin:.2f}</span><span>{vmax:.2f}</span>
      </div>
    </div>
    '''

dropdown = widgets.Dropdown(options=dropdown_options, value='sdcAre', description='Variabile:')
legend_out = widgets.Output()

initial_colors, vmin, vmax = get_colors(dropdown.value)
layer = lonboard.ScatterplotLayer.from_geopandas(
    centroids,
    get_fill_color=initial_colors,
    radius_min_pixels=1.5,
    radius_max_pixels=4,
)
m = lonboard.Map(layer)

def refresh_legend(column, vmin, vmax):
    legend_out.clear_output()
    with legend_out:
        display(widgets.HTML(legend_html(column, vmin, vmax)))

refresh_legend(dropdown.value, vmin, vmax)

def on_change(change):
    column = change['new']
    colors, vmin, vmax = get_colors(column)
    layer.get_fill_color = colors
    refresh_legend(column, vmin, vmax)

dropdown.observe(on_change, names='value')

display(widgets.VBox([dropdown, legend_out, m]))


## 3. Cluster analysis

Cluster analysis uses the data on form, function, and their combination in a K-Means clustering to identify homogenous patterns of built form.

### 3.1. Form

Clustering based on form uses contextualised characters as input.

In [ ]:
form = pd.read_parquet(folder + "convolutions.pq")
tess = gpd.read_parquet(folder + "tess.pq", columns=['tessellation', 'buildings', 'tID'])


In [ ]:
form = form.replace(np.inf, np.nan)  # e.g. due to tiny/edge tessellation cells


Raw data needs to be standardised. We use a standard scaler.

In [ ]:
scaler = preprocessing.StandardScaler()
data = scaler.fit_transform(form)


Any remaining missing values (e.g. at the edges of the study area) are replaced with 0.

In [ ]:
data[np.isnan(data)] = 0


We use clustergram to determine the optimal number of clusters, using 10 initialisations within a range from 1 to 30 clusters.

In [ ]:
cg_form = Clustergram(range(1, 30), backend='sklearn', n_init=10)
cg_form.fit(data)


Now we can plot the resulting clustergram. Due to outliers, we truncate the y axis to see cluster behaviour better &mdash; adjust the limits once you've inspected the actual range for Winterthur's data.

In [ ]:
ax = cg_form.plot(figsize=(20, 20), linewidth=0.5, cluster_style={"edgecolor": "r", "alpha": .6}, size=1,
       line_style={"alpha": .5})


Inspect the clustergram and pick the number of clusters where major branching settles down (10 for Singapore; expect a different number here). Before proceeding, we save the clustergram object to a file.

In [ ]:
with open(folder + 'clustergram_form.pickle', 'wb') as f:
    pickle.dump(cg_form, f)


Set `N_FORM_CLUSTERS` to the number you picked from the plot above, then see how the clusters look spatially.

In [ ]:
N_FORM_CLUSTERS = 12  # TODO: adjust based on the clustergram above
tess['clusters_form'] = cg_form.labels[N_FORM_CLUSTERS].values


In [ ]:
ax = tess.plot('clusters_form', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


In [ ]:
ax = tess.set_geometry('buildings').plot('clusters_form', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


### 3.2. Function

Clustering based on function uses only data on function, which we need to prepare.

In [ ]:
function_data = pd.read_parquet(folder + "func_data_tess.pq")


`use`, `transit_quality`, `urbanity` and `planning_typology` are all categorical, so they need to be encoded as boolean dummies.

In [ ]:
categorical_cols = ['use', 'transit_quality', 'urbanity', 'planning_typology']
dummies = pd.get_dummies(function_data[categorical_cols])

Continuous values need to be standardised.

In [ ]:
scaler = preprocessing.StandardScaler()
f_data = scaler.fit_transform(function_data.drop(columns=categorical_cols))

In [ ]:
f_data.shape


We merge everything back together.

In [ ]:
f_data = np.hstack([f_data, dummies.values])

In [ ]:
f_data.shape


In [ ]:
np.isnan(f_data).any().any()


Any remaining missing values (e.g. cells the new official statistics layers don't cover, such as water/park enclosures with no tessellation) are replaced with 0, same as for form.

In [ ]:
f_data[np.isnan(f_data)] = 0

We use clustergram again.

In [ ]:
cg_function = Clustergram(range(1, 30), backend='sklearn', n_init=10)
cg_function.fit(f_data)


In [ ]:
ax = cg_function.plot(figsize=(20, 20), linewidth=0.5, cluster_style={"edgecolor": "r", "alpha": .6}, size=1,
       line_style={"alpha": .5})


Clustergram should be saved to a file.

In [ ]:
with open(folder + 'clustergram_function.pickle', 'wb') as f:
    pickle.dump(cg_function, f)


Again, pick the optimal number of clusters from the plot above.

In [ ]:
N_FUNCTION_CLUSTERS = 9  # TODO: adjust based on the clustergram above
tess['clusters_function'] = cg_function.labels[N_FUNCTION_CLUSTERS].values


In [ ]:
ax = tess.plot('clusters_function', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


In [ ]:
ax = tess.set_geometry('buildings').plot('clusters_function', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


### 3.3. Spatial Signatures

Final clustering combines both form and function into a single classification. First we stack both preprocessed arrays together and then run clustergram.

In [ ]:
ss_data = np.hstack([data, f_data])


In [ ]:
cg_ss = Clustergram(range(1, 30), backend='sklearn', n_init=10, random_state=0)
cg_ss.fit(ss_data)


In [ ]:
c = ugg.COLORS
ax = cg_ss.plot(
    figsize=(12, 12),
    linewidth=0.5,
    cluster_style={"edgecolor": "w", "color": c[2], "linewidth": 1.5},
    size=1,
    line_style={"color": c[1]},
    pca_kwargs=dict(random_state=0),
)
plt.savefig(folder + "cgram.png", dpi=300, bbox_inches="tight")


In [ ]:
cg_labels = cg_ss.labels
cg_labels.columns = cg_labels.columns.astype(str)
cg_labels.to_parquet(folder + 'clustergram_ss_labels.pq')


Pick the optimal number of clusters from the plot above (Singapore settled on 12), then run the final clustering with many initialisations and a fixed random state for reproducibility.

In [ ]:
N_SS_CLUSTERS = 12  # TODO: adjust based on the clustergram above
%time km = MiniBatchKMeans(n_clusters=N_SS_CLUSTERS, n_init=1000, random_state=12345, batch_size=100000).fit(ss_data)

In [ ]:
tess['clusters'] = km.labels_


In [ ]:
ax = tess.plot('clusters', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


In [ ]:
ax = tess.set_geometry('buildings').plot('clusters', categorical=True, legend=True, figsize=(20, 20), cmap='tab20')
ax.set_axis_off()


In [ ]:
tess[['tID', 'clusters', 'clusters_form', 'clusters_function']].to_parquet(folder + "cluster_labels.pq")
tess[['tID', 'clusters']].to_csv(folder + "FINAL_cluster_labels.csv")


#### Generate signatures

As a final step, we generate a geometry representing spatial signatures as a combination of contiguous tessellation cells belonging to a single cluster. Since we have to dissolve a large number of polygons, we use `dask_geopandas` to parallelise the procedure. First we create a local cluster.

In [ ]:
client = Client(LocalCluster(n_workers=8))
client


We read only the `tessellation` column and cluster labels. The resulting geometry will no longer be enclosed tessellation, so we rename it to a generic `geometry` already.

In [ ]:
tess = gpd.read_parquet(folder + "tess.pq", columns=['tessellation']).rename_geometry("geometry")
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)


In [ ]:
tess['cluster'] = clusters.clusters.values


`dask_dissolve` mimics the behaviour of `geopandas.dissolve`, based on a parallel implementation using `dask-geopandas`.

In [ ]:
import dask
dask.config.set({"dataframe.shuffle.method": "tasks"})


In [ ]:
%%time
ddf = dask_geopandas.from_geopandas(tess.sort_values('cluster'), npartitions=64)
final = dask_dissolve(ddf, by='cluster').compute()


In [ ]:
final.plot('cluster', categorical=True, figsize=(20, 20), cmap='tab20')


Finally, we explode the geometries to get multiple signatures of the same kind instead of a single multipolygon. (The initial `reset_index` avoids a geopandas bug.)

In [ ]:
final.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True).to_parquet(folder + "signatures.pq")


Before we end, we close the dask client.

In [ ]:
client.close()


# 4. Figures

Finally, we create the final figures. Singapore's notebook hardcodes basemap tile bounds (`ax.set_xlim(11530000, 11590000)`) specific to Singapore's extent; here we derive the bounds from the data instead so this works for any place shape.

Note on basemaps: the original notebook uses `urbangrammar_graphics.get_tiles(...)`, which requires a personal Mapbox access token (the `token = ""` placeholder below is non-functional as-is, same as in the Singapore notebook). If you don't have a Mapbox token, swap `contextily.add_basemap(..., source=ugg.get_tiles(...))` for a free provider, e.g. `contextily.add_basemap(ax, source=contextily.providers.CartoDB.Positron)`.

In [ ]:
final = gpd.read_parquet(folder + "signatures.pq")


In [ ]:
enc = gpd.read_parquet(folder + "enclosures.pq")


In [ ]:
sns.set(context="paper", style="ticks", rc={'patch.force_edgecolor': False})


In [ ]:
cmap = ugg.get_colormap(final.cluster.nunique() + 1, randomize=False)


In [ ]:
gradient = np.linspace(0, 1, 256)
gradient = np.vstack((gradient, gradient))

fig, ax = plt.subplots(figsize=(8, .5))
ax.imshow(gradient, aspect='auto', cmap=cmap)


Symbology mapping cluster id to a colour. Adjust the number of entries to match `N_SS_CLUSTERS` chosen above.

In [ ]:
cols = cmap.colors
symbology = {i: cols[i] for i in range(final.cluster.nunique())}


In [ ]:
df = final.set_crs(enc.crs).to_crs(3857)


In [ ]:
token = ""  # requires a personal Mapbox access token; see note above for a token-free alternative


In [ ]:
minx, miny, maxx, maxy = df.total_bounds

ax = df.plot(color=df['cluster'].map(symbology), figsize=(20, 20), zorder=1, linewidth=.3, edgecolor='w', alpha=1)
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
if token:
    contextily.add_basemap(ax, crs=df.crs, source=ugg.get_tiles('roads', token), zorder=2, alpha=.3)
    contextily.add_basemap(ax, crs=df.crs, source=ugg.get_tiles('labels', token), zorder=3, alpha=1)
    contextily.add_basemap(ax, crs=df.crs, source=ugg.get_tiles('background', token), zorder=-1, alpha=1)
else:
    contextily.add_basemap(ax, crs=df.crs, source=contextily.providers.CartoDB.Positron, zorder=-1, alpha=1)
ax.set_axis_off()

scalebar = ScaleBar(dx=1,
                    color=ugg.COLORS[0],
                    location='lower right',
                    height_fraction=0.002,
                    pad=.5,
                    frameon=False,
                    )
ax.add_artist(scalebar)

ugg.north_arrow(plt.gcf(), ax, 0, size=.026, linewidth=1, color=ugg.COLORS[0], loc="upper left", pad=.002, alpha=.9)

from matplotlib.lines import Line2D
custom_points = [Line2D([0], [0], marker="o", linestyle="none", markersize=10, color=color) for color in symbology.values()]
leg_points = ax.legend(custom_points, symbology.keys(), loc='upper right', frameon=True)
ax.add_artist(leg_points)
plt.savefig(folder + "signatures.png", dpi=300, bbox_inches="tight")


In [ ]:
# Mappa: clustering solo "function"
# (pre-existing bug from an earlier session: `final_function` was referenced here
# but never defined/dissolved anywhere in the notebook -- built properly below
# from the persisted per-cell cluster labels instead of a dissolved geometry.)
import lonboard
from lonboard.colormap import apply_categorical_cmap
import matplotlib.cm as cm

cluster_labels = pd.read_parquet(folder + "cluster_labels.pq")
final_function = gpd.read_parquet(folder + "tess.pq", columns=["tID", "tessellation"]).merge(
    cluster_labels[["tID", "clusters_function"]], on="tID"
)
final_function_wgs84 = final_function.set_geometry("tessellation").rename_geometry("geometry").to_crs(4326)

values = sorted(final_function_wgs84['clusters_function'].unique())
cmap = cm.get_cmap('tab20', len(values))
color_lookup = {v: [int(x * 255) for x in cmap(i)[:3]] for i, v in enumerate(values)}
colors = apply_categorical_cmap(final_function_wgs84['clusters_function'], color_lookup)

layer = lonboard.PolygonLayer.from_geopandas(
    final_function_wgs84[['geometry']],
    get_fill_color=colors,
    get_line_color=[80, 80, 80, 255],
    get_line_width=20,
    opacity=0.5,
)
lonboard.Map(layer)

In [ ]:
tess = gpd.read_parquet(folder + "tess.pq").set_geometry('buildings')
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)
tess['cluster'] = clusters.clusters.values


In [ ]:
tess_wm = tess.to_crs(3857)
minx, miny, maxx, maxy = tess_wm.total_bounds

ax = tess_wm.plot(color=tess_wm['cluster'].map(symbology), figsize=(20, 20), zorder=1, alpha=1)
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
if token:
    contextily.add_basemap(ax, crs=df.crs, source=ugg.get_tiles('roads', token), zorder=2, alpha=.3)
    contextily.add_basemap(ax, source=ugg.get_tiles('labels', token), zorder=3, alpha=1)
    contextily.add_basemap(ax, source=ugg.get_tiles('background', token), zorder=-1, alpha=1)
else:
    contextily.add_basemap(ax, source=contextily.providers.CartoDB.Positron, zorder=-1, alpha=1)
ax.set_axis_off()

scalebar = ScaleBar(dx=1,
                    color=ugg.COLORS[0],
                    location='lower right',
                    height_fraction=0.002,
                    pad=.5,
                    frameon=False,
                    )
ax.add_artist(scalebar)

ugg.north_arrow(plt.gcf(), ax, 0, size=.026, linewidth=1, color=ugg.COLORS[0], loc="upper left", pad=.002, alpha=.9)

from matplotlib.lines import Line2D
custom_points = [Line2D([0], [0], marker="o", linestyle="none", markersize=10, color=color) for color in symbology.values()]
leg_points = ax.legend(custom_points, symbology.keys(), loc='upper right', frameon=True)
ax.add_artist(leg_points)
plt.savefig(folder + "signatures_buildings.png", dpi=300, bbox_inches="tight")


In [ ]:
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)


In [ ]:
counts = clusters.clusters.value_counts(normalize=True)
color = counts.index.map(symbology)
counts.plot.bar(color=color)
plt.ylabel("ratio")
plt.savefig(folder + "counts.pdf", dpi=300, bbox_inches="tight")


## 5. Cluster summary

We can also export summary values of our input characters for each cluster.

In [ ]:
data = gpd.read_parquet(folder + "data.pq")


We list the characters we want to summarise.

In [ ]:
characters = [
    "stcOri",
    "stbOri",
    "stbCeA",
    "sdcLAL",
    "sdcAre",
    "sscCCo",
    "sscERI",
    "sicCAR",
    "mtbAli",
    "mtbNDi",
    "mtcWNe",
    "mdcAre",
    "ltbIBD",
    "stcSAl",
    "ltcWRE",
    "sdbAre",
    "sdbPer",
    "sdbCoA",
    "ssbCCo",
    "ssbCor",
    "ssbSqu",
    "ssbERI",
    "ssbElo",
    "ssbCCM",
    "ssbCCD",
    "mtbSWR",
    "ldbPWL",
    "stbSAl",
    "libNCo",
    "sdsLen",
    "sdsSPW",
    "sdsSPO",
    "sdsSWD",
    "sssLin",
    "sdsAre",
    "sisBpM",
    "misRea",
    "mdsAre",
    "ldsMSL",
    "degree",
    "meshedness",
    "proportion_3",
    "proportion_4",
    "proportion_0",
    "local_closeness",
    "ldsCDL",
    "xcnSCl",
    "mtdMDi",
    "lddNDe",
    "linWID",
    "sddAre",
    "ldeAre",
    "ldePer",
    "lseCCo",
    "lseERI",
    "lseCWA",
    "lteOri",
    "lteWNB",
    "lieWCe",
]


In [ ]:
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)


In [ ]:
data["cluster"] = clusters.clusters.values


We get the mean of each character for each cluster.

In [ ]:
means = data[characters].groupby(data.cluster).mean().T


It is better to use full names, as the result should be interpretable. Codes are not that useful on their own.

In [ ]:
char_names = {
    'sdbAre': 'area of building',
    'sdbPer': 'perimeter of building',
    'sdbCoA': 'courtyard area of building',
    'ssbCCo': 'circular compactness of building',
    'ssbCor': 'corners of building',
    'ssbSqu': 'squareness of building',
    'ssbERI': 'equivalent rectangular index of building',
    'ssbElo': 'elongation of building',
    'ssbCCM': 'centroid - corner mean distance of building',
    'ssbCCD': 'centroid - corner distance deviation of building',
    'stbOri': 'orientation of building',
    'sdcLAL': 'longest axis length of ETC',
    'sdcAre': 'area of ETC',
    'sscCCo': 'circular compactness of ETC',
    'sscERI': 'equivalent rectangular index of ETC',
    'stcOri': 'orientation of ETC',
    'sicCAR': 'covered area ratio of ETC',
    'stbCeA': 'cell alignment of building',
    'mtbAli': 'alignment of neighbouring buildings',
    'mtbNDi': 'mean distance between neighbouring buildings',
    'mtcWNe': 'perimeter-weighted neighbours of ETC',
    'mdcAre': 'area covered by neighbouring cells',
    'ltcWRE': 'weighted reached enclosures of ETC',
    'ltbIBD': 'mean inter-building distance',
    'sdsSPW': 'width of street profile',
    'sdsSWD': 'width deviation of street profile',
    'sdsSPO': 'openness of street profile',
    'sdsLen': 'length of street segment',
    'sssLin': 'linearity of street segment',
    'ldsMSL': 'mean segment length within 3 steps',
    'degree': 'node degree of junction',
    'meshedness': 'local meshedness of street network',
    'proportion_3': 'local proportion of 3-way intersections of street network',
    'proportion_4': 'local proportion of 4-way intersections of street network',
    'proportion_0': 'local proportion of cul-de-sacs of street network',
    'local_closeness': 'local closeness of street network',
    'ldsCDL': 'local cul-de-sac length of street network',
    'xcnSCl': 'square clustering of street network',
    'mtdMDi': 'mean distance to neighbouring nodes of street network',
    'lddNDe': 'local node density of street network',
    'linWID': 'local degree weighted node density of street network',
    'stbSAl': 'street alignment of building',
    'stcSAl': 'street alignment of ETC',
    'mtbSWR': 'shared walls ratio of buildings',
    'sddAre': 'area covered by node-attached ETCs',
    'sdsAre': 'area covered by edge-attached ETCs',
    'sisBpM': 'buildings per meter of street segment',
    'mdsAre': 'reached area by neighbouring segments',
    'ldeAre': 'area of enclosure',
    'ldePer': 'perimeter of enclosure',
    'lseCCo': 'circular compactness of enclosure',
    'lseERI': 'equivalent rectangular index of enclosure',
    'lseCWA': 'compactness-weighted axis of enclosure',
    'lteOri': 'orientation of enclosure',
    'lteWNB': 'perimeter-weighted neighbours of enclosure',
    'lieWCe': 'area-weighted ETCs of enclosure',
    'ldbPWL': 'perimeter wall length of adjacent buildings',
    'libNCo': 'number of courtyards within adjacent buildings',
    'misRea': 'cells reached within neighbouring street segments',
}


In [ ]:
means.index = means.index.map(char_names)


We do the same with function characters.

In [ ]:
function_data = pd.read_parquet(folder + "func_data_tess.pq")


Function columns for Winterthur: `use`, `transit_quality`, `urbanity` and `planning_typology` (all categorical, excluded here), plus every numeric function layer from Section 2 (POI accessibility, street betweenness, population/employment/building-stock statistics, ...). NDVI/night lights are still TODO (Section 2.11-2.12) and are absent until implemented.

In [ ]:
f_cols = [
    c
    for c in (
        [
            "park_distance", "open_space_distance", "eating", "supermarkets", "monuments", "betweenness",
            "population", "frau_p", "ausland_p", "altersquot", "jugendquot", "gesamtquot",
            "jobs_establishments", "jobs_employed", "jobs_fte", "besch_f_p", "hightech_p", "wissdl_p",
        ]
        + sorted(c for c in function_data.columns if c.startswith("noga08_"))
        + [
            "buildings_n", "dwellings_n", "living_area_sqm",
            "bj_v45_p", "bj_46_80_p", "bj_81_00_p", "bj_n01_p",
            "efh_p", "mfh_p", "mischn_p", "anderen_p",
            "stwk_1_2_p", "stwk_3_4_p", "stwk_5_6_p", "stwk_ab7_p",
            "ndvi", "lights",
        ]
    )
    if c in function_data.columns
]
f_means = function_data[f_cols].groupby(data.cluster.values).mean().T

In [ ]:
rename = {
    "park_distance": "distance to nearest park",
    "open_space_distance": "distance to nearest open/green space",
    "eating": "eating establishments",
    "supermarkets": "supermarkets",
    "monuments": "historical monuments",
    "betweenness": "street betweenness centrality",
    "population": "population",
    "frau_p": "% female",
    "ausland_p": "% foreign origin",
    "altersquot": "old-age dependency ratio",
    "jugendquot": "youth dependency ratio",
    "gesamtquot": "total dependency ratio",
    "jobs_establishments": "business establishments",
    "jobs_employed": "jobs",
    "jobs_fte": "jobs (FTE)",
    "besch_f_p": "% female employees",
    "hightech_p": "% high-tech jobs",
    "wissdl_p": "% knowledge-intensive service jobs",
    "buildings_n": "buildings",
    "dwellings_n": "dwellings",
    "living_area_sqm": "living area (sqm)",
    "bj_v45_p": "% buildings built before 1945",
    "bj_46_80_p": "% buildings built 1946-1980",
    "bj_81_00_p": "% buildings built 1981-2000",
    "bj_n01_p": "% buildings built after 2001",
    "efh_p": "% single-family houses",
    "mfh_p": "% multi-family houses",
    "mischn_p": "% mixed-use buildings",
    "anderen_p": "% other building types",
    "stwk_1_2_p": "% buildings with 1-2 floors",
    "stwk_3_4_p": "% buildings with 3-4 floors",
    "stwk_5_6_p": "% buildings with 5-6 floors",
    "stwk_ab7_p": "% buildings with 7+ floors",
    "ndvi": "NDVI",
    "lights": "night lights",
}


def _prettify_noga(col):
    if col.startswith("noga08_") and col.endswith("_p"):
        return f"% jobs in NOGA08 sector {col[len('noga08_'):-len('_p')].upper()}"
    return rename.get(col, col)


f_means.index = [_prettify_noga(c) for c in f_cols]

Finally, we join form and function summaries and plot the final output as a styled pandas DataFrame.

In [ ]:
together = pd.concat([means, f_means])


In [ ]:
together.replace(np.inf, np.nan).fillna(0).style.format('{:.2f}').background_gradient(cmap=sns.diverging_palette(235, 16, 60, 55, as_cmap=True), axis=1)


## Create a data product

Create files that could be uploaded to a public repository (e.g. figshare) for public access: the raw measured data at the enclosed-tessellation-cell level, and the signature geometry.

In [ ]:
form = gpd.read_parquet(folder + "data.pq")
function = pd.read_parquet(folder + "func_data_tess.pq")
conv = pd.read_parquet(folder + "convolutions.pq")


In [ ]:
function


In [ ]:
combined = pd.concat([form, conv, function.drop(columns=["tID"])], axis=1)


In [ ]:
clusters = pd.read_csv(folder + "FINAL_cluster_labels.csv", index_col=0)


In [ ]:
combined["signature_type"] = clusters.clusters.values


In [ ]:
combined.to_parquet(folder + "zurich_data.parquet")


In [ ]:
sig = gpd.read_parquet(folder + "signatures.pq")


In [ ]:
sig.crs = combined.crs


In [ ]:
sig = sig.rename(columns={"cluster": "signature_type"})


In [ ]:
sig.to_parquet(folder + "signatures.pq")
